# GMU — Monotone-Projects Nash Equilibrium

This notebook demonstrates the main algorithm (`find_global_NE_monotone_projects`) and the
step-by-step visualization.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from ne_solvers import find_global_NE_monotone_projects
from visualization import animate_outer_history, animate_outer_history_rates

## Define problem instance

In [11]:
a = np.array([10, 9, 8, 3, 2], dtype=float)   # project parameters (nonincreasing)
r_total = np.array([15, 14, 2, 1], dtype=float) # player resources

## Run the monotone-projects algorithm

In [12]:
solver, marg, k, bc_penalty, d_rows, X, outer_hist = find_global_NE_monotone_projects(
    a,
    r_total,
    max_iter_inner=500,
    tol_row_inner=1e-6,
    bc_tol=1e-8,
    verbose=True,
    debug=False,                   # set True for per-step BC diagnostics
    store_X_in_history=True,       # needed for visualization
)

[init] k=[1, 1, 1, 1]  bc=8.835e+00  d_rows=0.000e+00
[step 1] expand_new j=0 -> k[j]=2  bc=7.767e+00
[step 1] expand_new j=1 -> k[j]=2  bc=7.689e+00
[step 1] expand_new j=2 -> k[j]=2  bc=7.688e+00
[step 1] expand_new j=3 -> k[j]=2  bc=7.688e+00
[step 2] expand_new j=0 -> k[j]=3  bc=2.611e+00
[step 2] expand_new j=1 -> k[j]=3  bc=2.560e+00
[step 2] expand_new j=2 -> k[j]=3  bc=2.559e+00
[step 2] expand_new j=3 -> k[j]=3  bc=2.559e+00
[step 3] expand_new j=0 -> k[j]=4  bc=1.519e+00
[step 3] expand_new j=1 -> k[j]=4  bc=1.514e+00
[step 3] expand_new j=2 -> k[j]=4  bc=1.514e+00
[step 3] refuse_new j=3 at K=4
[step 4] expand_new j=0 -> k[j]=5  bc=4.994e-01
[step 4] expand_new j=1 -> k[j]=5  bc=0.000e+00
[step 4] refuse_new j=2 at K=5
[done] step=4  k=[5, 5, 4, 3]  bc=0.000e+00  d_rows=1.212e-04


## Final result

In [13]:
print(f"Cutoff vector k : {k}")
print(f"BC penalty      : {bc_penalty:.3e}")
print(f"Row disruption  : {d_rows:.3e}")
print()
print("Allocation X (rows = players, cols = projects):")
print(np.round(X, 4))
print()
print("Player marginal rates c_rows:")
print(np.round(marg.c_rows, 6))

Cutoff vector k : [5, 5, 4, 3]
BC penalty      : 0.000e+00
Row disruption  : 1.212e-04

Allocation X (rows = players, cols = projects):
[[4.7371 4.2581 3.7791 1.3713 0.8545]
 [4.4367 3.9861 3.5355 1.2675 0.7742]
 [0.7688 0.6647 0.5606 0.0059 0.    ]
 [0.4029 0.3333 0.2638 0.     0.    ]]

Player marginal rates c_rows:
[[0.513396 0.513427 0.513518]
 [0.536732 0.536853 0.536757]
 [0.821683 0.821788 0.      ]
 [0.850112 0.       0.      ]]


## Step-by-step visualization — table animation

Each frame shows:
- **Zone header**: which projects belong to each zone
- **Allocation grid**: colored by player, numbers are investments x_{j,i}
- **Right columns**: player shadow rate c_j and payoff u_j
- **Bottom rows**: total load R per project, zone R and zone C

The highlighted column is the "frontier" project being considered at each step.

In [14]:
ani = animate_outer_history(
    outer_hist,
    a=a,
    r_players=r_total,
    interval_ms=900,
    figsize=(14, 6.5),
)
HTML(ani.data)  # display inline

## Step-by-step visualization — with rate line

Setting `rate_line=True` adds BEFORE/AFTER pairs that show:
- Player shadow rates c_j (filled circles, coloured by player)
- Candidate project entry rates q_i^{(j)}(0) (triangles, for the considered player)

This illustrates why the algorithm expands or refuses each player at each step.

In [6]:
ani_rates = animate_outer_history_rates(
    outer_hist,
    a=a,
    r_players=r_total,
    interval_ms=700,
    rate_line=True,    # show rate line with before/after pairs
    figsize=(14, 6.5),
)
HTML(ani_rates.data)

## Save frames to disk (for slides)

Uncomment to export PNG frames or an MP4.

In [ ]:
# from visualization import save_outer_history_rates_frames, save_outer_history_mp4

# Save individual PNG frames
# save_outer_history_rates_frames(
#     outer_hist, "gmu_frames",
#     a=a, r_players=r_total,
#     rate_line=False, dpi=200,
# )

# Save as MP4 (requires ffmpeg)
# save_outer_history_mp4(
#     outer_hist, "gmu.mp4",
#     a=a, r_players=r_total,
#     fps=6, dpi=180,
# )